Document Structure

In [1]:
from langchain_core.documents import Document
from langchain_text_splitters  import RecursiveCharacterTextSplitter

c:\Users\Asus\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.makedirs("../data/text_files", exist_ok=True)

In [3]:
sample_texts = {
    "../data/text_files/sample1.txt": "This is the content of sample text file 1.",
}

for file_path, content in sample_texts.items():
    with open(file_path, "w") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


Text Loader

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/sample1.txt")
doc = loader.load()

print(doc)

[Document(metadata={'source': '../data/text_files/sample1.txt'}, page_content='This is the content of sample text file 1.')]


C:\Users\Asus\AppData\Local\Temp\ipykernel_11484\1602887145.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Directory Loader

In [5]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)
docs = loader.load()

print(docs)

[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='This is the content of sample text file 1.'), Document(metadata={'source': '..\\data\\text_files\\sample2.txt'}, page_content='Artificial intelligence is transforming how we create, learn, and solve problems.\nBy analyzing data, recognizing patterns, and adapting to new situations, AI systems can help people make smarter decisions and build new kinds of applications.\nAs AI evolves, it opens possibilities for better tools, more efficient workflows, and a deeper understanding of how technology supports human creativity.')]


In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "../data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)
docs = loader.load()

print(docs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-02T15:25:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-02T15:25:28+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\About_ChatGPT_500_Words.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='About ChatGPT\nI am ChatGPT, an artificial intelligence language model created by OpenAI. My purpose is to assist\npeople by answering questions, explaining concepts, generating ideas, helping with writing, solving\nprogramming problems, and supporting learning across a wide range of subjects. Rather than\nthinking or feeling as humans do, I generate responses by recognizing patterns in language learned\nduring training and by using the context provided in each conversation. One of my strengths is\nversatility. I can help students understand mathematics, science, liter

In [6]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(docs)
chunks

Split 2 documents into 2 chunks

Example chunk:
Content: This is the content of sample text file 1....
Metadata: {'source': '..\\data\\text_files\\sample1.txt'}


[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='This is the content of sample text file 1.'),
 Document(metadata={'source': '..\\data\\text_files\\sample2.txt'}, page_content='Artificial intelligence is transforming how we create, learn, and solve problems.\nBy analyzing data, recognizing patterns, and adapting to new situations, AI systems can help people make smarter decisions and build new kinds of applications.\nAs AI evolves, it opens possibilities for better tools, more efficient workflows, and a deeper understanding of how technology supports human creativity.')]

Embeddings And VectorDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
            if not self.model:
                raise ValueError("Model is not loaded.")

            # 1. Force into a list if a single string or single object is passed
            if isinstance(texts, str) or hasattr(texts, 'page_content') or hasattr(texts, 'text'):
                texts = [texts]
            
            clean_texts = []
            
            # 2. Flatten and parse the inputs
            def extract_text(item):
                if item is None:
                    return None
                # Handle standard LangChain Document
                if hasattr(item, 'page_content'):
                    return item.page_content
                # Handle LlamaIndex Document/Node
                if hasattr(item, 'text'):
                    return item.text
                # Handle dictionaries with common text keys
                if isinstance(item, dict):
                    return item.get('text', item.get('page_content', str(item)))
                return str(item)

            # Iterate and recursively flatten if a list of lists was accidentally passed
            for t in texts:
                if isinstance(t, (list, tuple, np.ndarray)):
                    for sub_t in t:
                        text_val = extract_text(sub_t)
                        if text_val and text_val.strip():
                            clean_texts.append(text_val.strip())
                else:
                    text_val = extract_text(t)
                    if text_val and text_val.strip():
                        clean_texts.append(text_val.strip())

            # 3. Check for empty text arrays
            if not clean_texts:
                try:
                    dim = self.model.get_embedding_dimension()
                except Exception:
                    return np.empty((0,))
                return np.empty((0, dim))

            print(f"Generating embeddings for {len(clean_texts)} texts...")
            
            # 4. Batch encoding
            batch_size = 64
            embeddings_batches = []
            for i in range(0, len(clean_texts), batch_size):
                batch = clean_texts[i:i+batch_size]
                # convert directly to numpy array to avoid overhead
                emb = self.model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
                embeddings_batches.append(emb)
                
            embeddings = np.vstack(embeddings_batches)
            return embeddings


Initialize the Embedding Manager

In [9]:
embedding_manager = EmbeddingManager()

Loading model all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2661.16it/s]


Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


Vector Store

In [10]:
class VectorStore:
    def __init__(self, collection_name: str = "document_embeddings", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "Collection of document embeddings"})

            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB client: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise

        print(f"Added {len(documents)} documents to the collection '{self.collection_name}'.")  


vector_store = VectorStore()
vector_store

ChromaDB collection 'document_embeddings' initialized successfully.
Existing documents in collection: 16


Convert Text to Embeddings

In [11]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 2 texts...


Store the embeddings in VectorDB

In [12]:
vector_store.add_documents(chunks, embeddings)

successfully added 2 documents to vector store
total documents in collection: 18
Added 2 documents to the collection 'document_embeddings'.


Retriever Pipeline From VectorStore

In [13]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieval(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top K = {top_k}, Score Threshold = {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print(f"No documents found")

            return retrieved_docs
        except Exception as e:
            print("No document retrieved")
            return []
rag_retriever = RAGRetriever(vector_store, embedding_manager)

rag_retriever

In [14]:
rag_retriever.retrieval("Who created ChatGPT?")

Retrieving documents for query: Who created ChatGPT?
Top K = 5, Score Threshold = 0.0
Generating embeddings for 1 texts...
Retrieved 0 documents (after filtering)


[]

Integration VectorDB Context Pipeline with LLM output

In [23]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=groq_api_key, model="gemma2-9b-it", temperature=0.1, max_tokens=1024)


def rag_simple(query, retriever, llm, top_k=5, score_threshold=0.0):
    retrieved_docs = retriever.retrieval(query, top_k=top_k, score_threshold=score_threshold)

    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else "No relevant documents found."

    if not context.strip():
        context = "No relevant documents found."
    
    prompt = f"Answer the following question based on the provided context:\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content

In [24]:
pass